# Seminar 14: Advanced Diffusion

Date: 2025-04-22

In this seminar we explore Hugging Face **[Diffusers](https://huggingface.co/docs/diffusers/index)**. We begin with the math of diffusion, then build, tweak and extend pipelines for text‑to‑image, img2img, inpainting, LoRA adapters and textual inversion.

## 1.1  Device detection

In [ ]:
import torch
from PIL import Image

def get_torch_device() -> torch.device:
    """Return the best available torch device (CUDA > MPS > CPU).
    
    Returns
    -------
    torch.device
        * `torch.device("cuda")` if NVIDIA GPU is available.
        * `torch.device("mps")` if Apple-Silicon GPU is available.
        * `torch.device("cpu")` otherwise.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE: torch.device = get_torch_device()
print(f"Running on {DEVICE}.")

Libraries installation:
```bash
%pip install -qU "torch>=2.2" "accelerate>=0.30.0" "diffusers[torch]>=0.33.0" "compel>=2.0.1" "huggingface_hub>=0.22.2" "pillow>=10.3" "gradio>=4.28" "peft>=0.10.0" "hf_xet"
```

## 2  Diffusion‑model fundamentals  

We recall the *Denoising Diffusion Probabilistic Model* (DDPM).

### 2.1  Forward process  

At each step $t$ we add Gaussian noise  

$$
q(\mathbf x_t \mid \mathbf x_{t-1}) \;=\; \mathcal N\!\bigl(
    \mathbf x_t;\;
    \sqrt{1-\beta_t}\,\mathbf x_{t-1},\,
    \beta_t \mathbf I
\bigr),
\qquad t = 1,\dots,T,
$$

with a pre‑defined variance schedule $\{\beta_t\}_{t=1}^T$.

### 2.2  Reverse process and objective  

The learnable model $p_\theta$ predicts either $\mathbf x_{t-1}$ or the added noise $\boldsymbol\epsilon$.  The **“simple” loss** used by *Diffusers* is

$$
\mathcal L_\text{simple}(\theta) \;=\;
\mathbb E_{t,\mathbf x_0,\boldsymbol\epsilon}
\bigl[\;
\lVert \boldsymbol\epsilon - \boldsymbol\epsilon_\theta(\mathbf x_t,t) \rVert_2^2
\bigr].
$$

For detailed derivations see [Ho et al., 2020].

## 3 Text‑to‑Image (Stable Diffusion, Latent‑Space Formulation)

Given a text prompt $\mathbf p$, the pipeline:

1. **Embeds text** with CLIP, yielding embedding $e_\mathbf p\in\mathbb R^{77\times768}$.
2. **Samples a latent** $\mathbf z_T\sim\mathcal N(\mathbf 0,\mathbf I)$ in $\mathbb R^{4\times H/8\times W/8}$.
3. **Iteratively denoises** $\mathbf z_t\!\to\!\mathbf z_{t-1}$ with a U‑Net $\epsilon_\theta$ trained on the *noise‑prediction* objective  
   $$
   \mathcal L_\text{simple}
     =\mathbb E_{t,\mathbf x_0,\boldsymbol\epsilon}
       \!\left[\,\bigl\lVert\boldsymbol\epsilon-
       \boldsymbol\epsilon_\theta(\mathbf z_t,t,e_\mathbf p)\bigr\rVert_2^2\right].
   $$
4. **Applies classifier‑free guidance** (CFG) at every step  
   $$
   \boldsymbol\epsilon_\text{cfg}
     =\boldsymbol\epsilon_\text{uncond}
      +s\bigl(\boldsymbol\epsilon_\text{cond}-\boldsymbol\epsilon_\text{uncond}\bigr),
   $$
   where $s=\texttt{guidance\_scale}$ controls prompt strength.
5. **Decodes** $\mathbf z_0$ to pixels with a V‑AE decoder $D \colon \mathbb R^{4\times\!H/8\times W/8}\!\to\!\mathbb R^{3\times H\times W}$.

Because diffusion runs in latent space, memory and compute scale with $(H/8)(W/8)$ not $HW$.

In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-2-1-base"

In [ ]:
from diffusers import AutoPipelineForText2Image

pipe_txt2img = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
).to(DEVICE)

pipe_txt2img.enable_attention_slicing()   # lower‑memory inference

prompt: str = (
    "cinematic photograph of Godzilla eating sushi with a cat in an izakaya, "
    "35 mm film, professional, 4K, highly detailed"
)

image = pipe_txt2img(prompt, num_inference_steps=25, guidance_scale=7.5).images[0]
image

### 3.1  What did we just do?  

1. *Loaded* the base SD‑2.1 weights in float32.  
2. *Moved* them to GPU/MPS/CPU.  
3. *Generated* an image in ≈ 25 denoising steps.  

> **Tip** `pipe.enable_attention_slicing()` trades inference speed for smaller memory footprints.

## 4 Schedulers (Euler‑Discrete, DPM‑Solver++, DDIM)

Diffusion sampling solves the *reverse* ODE  

$$
\frac{\mathrm d\mathbf z_t}{\mathrm dt}
  =f_\theta(\mathbf z_t,t):=
   -\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,
    \boldsymbol\epsilon_\theta(\mathbf z_t,t),
$$  

where $\beta_t$ is the variance schedule and $\bar\alpha_t=\prod_{s\le t}(1-\beta_s)$.

### Euler‑Discrete

A first‑order explicit solver with step $\Delta t$  

$$
\mathbf z_{t-\Delta t}
  =\mathbf z_t+f_\theta(\mathbf z_t,t)\,\Delta t
   +\sigma_t\sqrt{\Delta t}\,\mathbf\varepsilon,\qquad
   \mathbf\varepsilon\sim\mathcal N(\mathbf 0,\mathbf I). \tag{1}
$$  

Fast (20–30 steps) and robust on photorealistic prompts.

### DPM‑Solver++

Observes that the diffusion ODE is *semi‑linear* and constructs a second‑/third‑order multistep update that eliminates linear‑term error:  

$$
\mathbf z_{t_{k-1}}
  =\sum_{i=0}^{m-1}
     \omega_i\,
     \mathbf z_{t_{k-i}}
   +\eta_k\,\boldsymbol\epsilon_\theta(\mathbf z_{t_{k-1}},t_{k-1}),
$$  

with adaptive $\omega_i,\eta_k$ pre‑computed from $(\beta_t)$.  
Converges in 10–15 steps with quality close to ancestral sampling.

### DDIM

A deterministic variant that integrates the **same** ODE with no stochastic term, allowing *perfectly repeatable* edits and acceleration via *parallel* sampling.

In [ ]:
from diffusers import DiffusionPipeline, EulerDiscreteScheduler, DPMSolverMultistepScheduler
import numpy as np

def swap_scheduler(
    pipe: DiffusionPipeline,
    scheduler_cls: type[EulerDiscreteScheduler | DPMSolverMultistepScheduler],
) -> DiffusionPipeline:
    """Return a *new* pipeline whose scheduler is replaced in-place.

    Parameters
    ----------
    pipe
        Original pipeline.
    scheduler_cls
        Scheduler class from ``diffusers.schedulers``.

    Returns
    -------
    diffusers.DiffusionPipeline
        Same object with updated `.scheduler`.
    """
    pipe.scheduler = scheduler_cls.from_config(pipe.scheduler.config)
    return pipe

pipe_euler = swap_scheduler(pipe_txt2img, EulerDiscreteScheduler)
pipe_dpm   = swap_scheduler(pipe_txt2img, DPMSolverMultistepScheduler)

prompt_sched: str = "watercolour sketch of Mount Fuji at sunrise"
imgs = [
    pipe_euler(prompt_sched, num_inference_steps=35).images[0],
    pipe_dpm(prompt_sched,   num_inference_steps=35).images[0],
]


side_by_side = Image.fromarray(
    np.concatenate([np.asarray(im) for im in imgs], axis=1)
)
side_by_side

*Left: Euler Discretised — Right: DPM‑Solver++*  
Schedulers strongly influence final aesthetics **and** speed.

## 5 Batch Generation & Image Grids

In [ ]:
from typing import Sequence

def image_grid(imgs: Sequence[Image.Image], rows: int, cols: int) -> Image.Image:
    """Tile images into a single composite grid.

    Parameters
    ----------
    imgs
        Sequence of PIL images, length ≥ ``rows*cols``.
    rows, cols
        Grid layout.

    Returns
    -------
    PIL.Image.Image
        Composite RGB image.
    """
    w, h = imgs[0].size
    grid = Image.new("RGB", (cols * w, rows * h))
    for idx, im in enumerate(imgs[: rows * cols]):
        grid.paste(im, (idx % cols * w, idx // cols * h))
    return grid

prompts = [
    "red sports car at night on a wet road",
    "yellow sports car at night on a wet road",
    "blue sports car at night on a wet road",
]
batch_imgs = pipe_txt2img(prompts, num_inference_steps=20).images
image_grid(batch_imgs, rows=1, cols=3)

## 6 Prompt Engineering with **Compel**

Let a prompt contain tokens $\{\tau_i\}$ with weights $\{w_i\}$.  
Compel parses the string syntax `(token:weight)` and constructs the *weighted* CLIP embedding  

$$
\mathbf e_\text{prompt}
  =\frac{\sum_i w_i\,E(\tau_i)}
         {\left\lVert\sum_i w_i\,E(\tau_i)\right\rVert_2},
$$  

where $E$ is the frozen text‑encoder.  
Thus the gradient wrt a concept scales linearly with $w_i$.

- **Boost** concepts: `(sun:1.5)` multiplies its embedding by 1.5.  
- **Suppress**: `(crowd:0.2)` down‑weights distractions.  

The weighted embedding is fed straight into the diffusion U‑Net, giving sub‑token control without retraining.


In [ ]:
from typing import List

from compel import Compel

compel = Compel(
    tokenizer=pipe_txt2img.tokenizer,
    text_encoder=pipe_txt2img.text_encoder,
)

prompts = [
    "a cat playing with a ball-- in the forest",
    "a cat playing with a ball++ in the forest",
    "a cat playing with a ball++ in the forest---",
]

prompt_embeds: torch.FloatTensor = compel(prompts)

batch_imgs: List[Image.Image] = pipe_txt2img(
    prompt_embeds=prompt_embeds,
    num_inference_steps=30,
).images

image_grid(batch_imgs, rows=1, cols=3)

## 7 Image‑to‑Image (Latents‑Respace)

Given a reference image $\mathbf x_0$ and strength $\lambda\!\in[0,1]$:

1. **Encode** to latent $\mathbf z_0 = E(\mathbf x_0)$.  
2. **Add noise** to timestep $t=\lfloor\lambda T\rfloor$  
   $$
   \mathbf z_t
     =\sqrt{\bar\alpha_t}\,\mathbf z_0
      +\sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon,
      \qquad\boldsymbol\epsilon\sim\mathcal N(\mathbf 0,\mathbf I).
   $$
3. **Denoise** $\mathbf z_t\!\to\!\mathbf z_0'$ with a *new* prompt.  
4. **Decode** $D(\mathbf z_0')$ to obtain the edited image.

Low $\lambda$ preserves composition; high $\lambda$ drifts toward full synthesis.

In [ ]:
from diffusers.utils import load_image
from typing import Tuple
from PIL import Image

def load_image_url(url: str, size: Tuple[int, int] | None = None) -> Image.Image:
    """Download a remote image and convert to RGB.

    Parameters
    ----------
    url
        Direct or redirecting HTTPS URL to an image on the Hugging Face Hub (or elsewhere).
    size
        Optional (width, height) to `.resize()` the image.

    Returns
    -------
    PIL.Image.Image
        RGB image; ready for Diffusers pipelines.
    """
    img: Image.Image = load_image(url)          # handles redirects + converts to RGB
    return img.resize(size) if size else img

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

init_image = load_image_url(
    "https://hf.co/datasets/diffusers/diffusers-images-docs/resolve/main/mountain.png",
    size=(768, 768),
)

pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32
).to(DEVICE)

# Strength = 0.6 -- 60% of inference steps

stylised = pipe_img2img(
    prompt="forest owl, pencil drawing",
    image=init_image,
    strength=0.6,
    num_inference_steps=40,
).images[0]

stylised

In [ ]:
# Strength = 1.0 is a full denoising pass

stylised = pipe_img2img(
    prompt="forest owl, pencil drawing",
    image=init_image,
    strength=1.0,
    num_inference_steps=40,
).images[0]
stylised

## 8 Inpainting (Diffusion with Spatial Mask)

Let $M\in\{0,1\}^{H\times W}$ be a binary mask (\(1=\) *region to change*).

1. Encode image $\mathbf x_0$ to latent $\mathbf z_0$.
2. Apply forward diffusion to get $\mathbf z_t$.  
3. **Latent replacement each step** $t\!\to\!t-1$  

   $$
   \mathbf z_{t-1}
     =(1-M)\odot\mathbf z_\text{orig,\,t-1}
      +M\odot\mathbf z_{t-1}^*
   $$  

   where $\mathbf z_{t-1}^*$ is the scheduler’s update and  
   $\mathbf z_\text{orig,\,t-1}$ is the *noised original* latent.  
   Un‑masked pixels stay faithful while masked pixels iterate toward the prompt.

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

img_url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/billow926-12-Wc-Zgx6Y.png"
mask_url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/billow926-12-Wc-Zgx6Y_mask.png"

init_image = load_image_url(
    img_url,
    size=(768, 768),
)
mask_image = load_image_url(
    mask_url,
    size=init_image.size,
)

pipe_inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32
).to(DEVICE)

hole_filled = pipe_inpaint(
    prompt="Two red balls",
    image=init_image,
    mask_image=mask_image,
    num_inference_steps=50,
).images[0]
hole_filled

## 9 LoRA (Low‑Rank Adaptation)

A linear layer with weights $W\in\mathbb R^{d\times k}$ is *augmented* by  

$$
W' = W + \alpha\,\frac{1}{r}\,B\,A,
\quad
B\in\mathbb R^{d\times r},\;
A\in\mathbb R^{r\times k},\;
r\ll\min(d,k),
$$  

where only $A$ and $B$ are trained; $\alpha$ is a scale hyper‑parameter.

- **Parameter‑efficiency**: train $O(r(d+k))$ params (≈0.2 % for SD‑XL).  
- **Storage & speed**: at inference we either *add* the low‑rank term or permanently **fuse** it into $W$ (one GEMM).


In [ ]:
from diffusers import DiffusionPipeline

pipe_lora = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float32,
).to(DEVICE)

pipe_lora.load_lora_weights(
    "nerijs/pixel-art-xl",
    weight_name="pixel-art-xl.safetensors",
    adapter_name="pixel",
)
pipe_lora.fuse_lora(lora_scale=0.8)

prompt = "cyberpunk street scene, pixel‑art style"
pixel_art = pipe_lora(prompt, num_inference_steps=30).images[0]
pixel_art

## 10 Textual Inversion (Learning New Tokens)

Goal: teach the model a new concept from a *handful* of images $\{\mathbf x^{(j)}\}$.

1. **Instantiate** a learnable embedding $\mathbf v^*$ and map it to a special token `<S*>`.  
2. **Freeze** all U‑Net and CLIP weights; optimise $\mathbf v^*$ only:  

   $$
   \min_{\mathbf v^*}\;
   \mathbb E_{t,\;j,\;\boldsymbol\epsilon}
     \Bigl[\bigl\lVert\boldsymbol\epsilon-
       \boldsymbol\epsilon_\theta(\mathbf z_t^{(j)},t,e_{<S*>})\bigr\rVert_2^2\Bigr].
   $$
3. **Inference**: include `<S*>` in any prompt; its embedding steers generation toward the learned appearance.

Textual inversion therefore expands the *vocabulary* of a diffusion model without touching billions of U‑Net parameters.

In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float32
).to(DEVICE)

pipe.load_textual_inversion("sd-concepts-library/gta5-artwork")
img = pipe("sunset over the city skyline, <gta5-artwork>", num_inference_steps=30).images[0]
img